all nz CoastSat data, all transects, organized as follows.

For each transect and each CoastSat timestamp t (with previous timestamp t-1), I want the shoreline position/change paired with the integrated forcing (waves, and hopefully surge) accumulated over the interval [t_-1, t]. I’ll then build a seq2one model where inputs are previous position(s) + Δt + cumulative forcing, and the target is the next position (or Δposition).

If it’s feasible, I would like the NZ CoastSat data be exported/merged into 2 tables, structured like this:

A single Transect metadata csv 

a) transect_id (and beach/site id/name)

b) transect reference coordinate(s): lat/lon for the transect (e.g., transect origin or MSL intersection, whatever is easier)

c) beach-face slope for that transect (using the slope we already have)

d) wave measurement location (lat/lon - I was thinking ≥10 m depth)



A second 'flat' csv/table, where each observation is a single row:

a) trasnect_id (to merge with metadata table)

b) t = date/time of satellite shoreline measurement

c) x = shoreline position along the transect at t(k) (from CoastSat)

d) Δx = x − x_-1  (shoreline change since previous observation)

e) Δt = (t − t_-1) in days

Wave forcing matched to each transect 

f) over the interval [t_-1, t], provide:

   - a cumulative wave-energy (H^2), wave-power proxy (H^2*Tp)  and H^2*Tp

   - alternatively, the raw time series at that wave point so I can compute the cumulative metric myself



g) I’d also take simple summaries over [t-1, t] like mean and max Hsig, mean Tp and Tpeak, and mean direction (Dir/PkDir), but the key is the cumulative forcing over the interval.

h) Storm surge (optional, if doable) using the NZ surge dataset (lower spatial resolution is fine): pick the nearest grid point and provide an interval summary over [t-1, t]: max, mean and cumulative surge.



Time range

- I’m happy to truncate the merged dataset to the period where the wave hindcast exists, so every shoreline interval has forcing.



Output format

- two (or 1 merged) table (CSV would be ideal, but other format is fine too)

In [1]:
import pandas as pd
import geopandas as gpd
from glob import glob

In [2]:
transects = gpd.read_file("transects_extended.geojson")
transects

,id,site_id,orientation,along_dist,along_dist_norm,beach_slope,cil,ciu,trend,n_points,n_points_nonan,r2_score,mae,mse,rmse,intercept,ERODIBILITY,geometry
0,aus0001-0000,aus0001,104.347648,0.000000,0.000000,0.085,0.0545,0.2000,-1.441081,767.0,428.0,0.168420,28.102591,1263.560863,35.546601,179.085729,None,"LINESTRING (153.26555 -24.7007, 153.26938 -24...."
1,aus0001-0001,aus0001,93.495734,98.408334,0.002935,0.050,0.0387,0.0640,-1.037105,767.0,569.0,0.097874,25.419324,1033.770813,32.152306,212.247788,None,"LINESTRING (153.26525 -24.7019, 153.2692 -24.7..."
2,aus0001-0002,aus0001,82.069341,198.408334,0.005918,0.050,0.0428,0.0647,-0.680019,767.0,588.0,0.053927,22.632907,838.007507,28.948359,205.106151,None,"LINESTRING (153.26539 -24.70316, 153.26931 -24..."
3,aus0001-0003,aus0001,81.192757,298.402523,0.008900,0.055,0.0480,0.0659,-0.405198,767.0,598.0,0.023412,20.749758,698.653187,26.432048,191.745881,None,"LINESTRING (153.26555 -24.70408, 153.26945 -24..."
4,aus0001-0004,aus0001,81.065473,398.402523,0.011882,0.075,0.0614,0.0922,-0.090025,767.0,608.0,0.001277,19.889328,655.810616,25.608800,175.092121,None,"LINESTRING (153.2657 -24.70497, 153.26961 -24...."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
148189,nzd0604-0006,nzd0604,NaN,NaN,NaN,0.015,0.0137,0.0154,1.617348,354.0,340.0,0.099161,27.201442,1280.384719,35.782464,284.319742,None,"LINESTRING (175.17153 -36.94882, 175.1696 -36...."
148190,nzd0604-0007,nzd0604,NaN,NaN,NaN,0.010,0.0100,0.0114,-0.200530,354.0,348.0,0.004486,16.607071,481.482002,21.942698,336.186244,None,"LINESTRING (175.17234 -36.94851, 175.17041 -36..."
148191,nzd0604-0008,nzd0604,NaN,NaN,NaN,0.015,0.0146,0.0157,0.481883,354.0,346.0,0.029615,14.782436,412.410982,20.307904,289.699114,None,"LINESTRING (175.17356 -36.94778, 175.17058 -36..."
148192,nzd0604-0009,nzd0604,NaN,NaN,NaN,0.010,0.0100,0.0103,3.734130,354.0,333.0,0.334384,31.409671,1475.331264,38.410041,293.304520,None,"LINESTRING (175.1747 -36.9462, 175.1703 -36.94..."
